# 哨兵机器人自主决策行为树：自定义节点使用说明

> 本手册描述 `rmuc_2026.xml` 中出现的所有 **项目自定义节点**（非 BT.CPP 内置控制节点），给出端口、语义、返回值约定与实现要点。  
> ⚠️ 消息已从单一 `RMUC.msg` 拆分为 **12 个独立话题**（含 7 个 P0 新增话题），各自拥有独立订阅节点。详见 `RMUC_Msg_Split_Doc.ipynb`。  
> 📋 共计 **57 个自定义节点**：12 订阅 + 3 黑板初始化/解析 + 5 决策 + 4 导航动作 + 3 交战 + 1 机器人控制 + 8 导航选择 + 4 复活恢复 + 16 条件 + 1 装饰器。

## 1. 规则约束要点

> 用于理解行为树的策略设计背景。

| 主题 | 规则要点 |
|------|---------|
| **比赛阶段** | 七分钟比赛阶段（`stage_remain_time` 以秒计）为主循环执行条件；非比赛阶段回家并禁用发射 |
| **姿态系统** | 进攻/防御/移动三种姿态，切换冷却 **5 秒**；单局累计在某姿态超过 **3 分钟**，该姿态效果下降 |
| **脱战** | 存活状态下连续 **6 秒** 未发射弹丸且未被扣血；远程补血/远程补弹仅在脱战状态可用 |
| **补给区回血** | 占领己方补给区增益点获得每秒上限血量 **10%** 回血；开始 4 分钟后，若脱战且占领补给区，提升至每秒 **25%**（非脱战立即失效） |
| **允许发弹量** | 初始 **300**；可在补给区/基地增益点/前哨站增益点兑换；远程兑换成功后 **6 秒**生效；补给区每分钟占领一次累计 **+100 发**（可累积） |
| **复活与虚弱** | 正常读条复活后进入"虚弱"：发射机构锁定、无法占领增益点；接触可占领的前哨站/基地/补给区模块卡即可解除虚弱 |
| **己方堡垒** | 提供防御增益与射击热量冷却增益，以及"储备允许发弹量"；强度与 Δ（己方基地血量上限 − 现有基地血量）相关 |
| **敌方堡垒** | 比赛进行 3 分钟且对方前哨站被击毁后可占领；占领期间可能获得高易伤，属于 **高风险目标** |
| **增益系统 (NEW)** | 包含冷却增益、防御增益、易伤增益、能量增益等；由 `robot_buff` 话题实时获取百分比数值 |
| **复活机制 (NEW)** | 支持免费复活与付费立即复活；立即复活费用随累计次数递增（`cumulative_instant_count` 追踪） |

## 2. 黑板（Blackboard）关键字段约定

> `PerceptionAndBlackboard` 子树通过 **12 个独立话题** 订阅数据（含 7 个 P0 新增），再由 `ParseSentryBlackboard` 解析写入黑板。

| 字段 | 含义 | 来源话题 |
|:---|:---|:---|
| `{game_status}` | 原始比赛状态消息 | `/game_status` |
| `{robot_status}` | 原始机器人状态消息 | `/robot_status` |
| `{rfid_status}` | 原始 RFID 状态消息 | `/rfid_status` |
| `{radar_tracks}` | 原始雷达跟踪消息 | `/radar/enemy_tracks` |
| `{pose_x}` / `{pose_y}` / `{pose_yaw}` | 自身位姿 | `/robot_position` |
| `{pose}` | 完整位姿消息 | `/robot_position` |
| `{is_at_nav_goal}` | 是否到达导航目标 (bool) | `/robot_position` |
| `{now_ms}` | 当前时间戳 (ms) | `/game_status` |
| `{sentry_decision_status}` | 哨兵决策状态 **(NEW P0)** | `/sentry_decision_status` |
| `{robot_buff}` | 机器人增益信息 **(NEW P0)** | `/robot_buff` |
| `{projectile_allowance}` | 允许发弹量信息 **(NEW P0)** | `/projectile_allowance` |
| `{field_status}` | 场地状态信息 **(NEW P0)** | `/field_status` |
| `{enemy_mark}` | 敌方标记信息 **(NEW P0)** | `/enemy_mark` |
| `{team_positions}` | 队伍位置信息 **(NEW P0)** | `/team_positions` |
| `{team_hp}` | 队伍血量信息 **(NEW P0)** | `/team_hp` |
| --- | --- | --- |
| `{game.remain_s}` / `{game.elapsed_s}` | 比赛剩余/已进行时长（秒） | ParseSentryBlackboard |
| `{hp.cur}` / `{hp.max}` | 当前 / 上限血量 | ParseSentryBlackboard |
| `{heat.cur}` | 当前射击热量 | ParseSentryBlackboard |
| `{ammo.allow}` / `{ammo.left}` | 剩余允许发弹量 / 物理弹丸 | ParseSentryBlackboard |
| `{economy.coins}` | 队伍金币 | ParseSentryBlackboard |
| `{state.disengaged}` / `{state.disengage_cd_s}` | 是否脱战 / 脱战倒计时 | ParseSentryBlackboard |
| `{state.is_dead}` / `{state.is_weak}` | 是否战亡 / 虚弱 | ParseSentryBlackboard |
| `{state.respawn_invincible}` **(NEW P1)** | 复活无敌状态 | ParseSentryBlackboard |
| `{state.is_power_boosted}` **(NEW P1)** | 能量增强状态 | ParseSentryBlackboard |
| `{base.hp.cur}` / `{base.hp.max}` | 己方基地血量 | ParseSentryBlackboard |
| `{outpost.alive}` | 己方前哨站是否存活 | ParseSentryBlackboard |
| `{combat.has_target}` / `{combat.best_target}` | 目标检测 | ParseSentryBlackboard |
| `{threat.base}` | 基地威胁评估 | ParseSentryBlackboard |
| `{buff.cool_value}` **(NEW P0)** | 冷却增益数值 | ParseSentryBlackboard |
| `{buff.defense_pct}` **(NEW P0)** | 防御增益百分比 | ParseSentryBlackboard |
| `{buff.vulnerability_pct}` **(NEW P0)** | 易伤增益百分比 | ParseSentryBlackboard |
| `{enemy.hero_vuln}` ... `{enemy.sentry_vuln}` **(NEW P0)** | 敌方各兵种易伤状态 | ParseSentryBlackboard |
| `{team.hp.*}` **(NEW P0)** | 队伍各机器人血量 | ParseSentryBlackboard |
| `{field.*_status}` **(NEW P0)** | 场地各增益点状态 | ParseSentryBlackboard |
| `{fortress.ammo}` **(NEW)** | 堡垒储备弹药量 | ParseSentryBlackboard |
| `{cumulative_instant_count}` **(NEW P1)** | 累计立即复活次数 | ParseSentryBlackboard |

## 3. 自定义节点目录

> BT.CPP 内置节点（Sequence / Fallback / ReactiveSequence / ReactiveFallback / WhileDoElse 等）不在此列。  
> `RateController` 为装饰器节点，见第 3.14 节。

---

### 3.1 订阅节点（12 个）

- **类型**：Action（RosTopicSubNode，订阅并写黑板）
- **每个节点订阅独立话题**：

| # | 节点 | 话题 | 输出端口 |
|:-:|:---|:---|:---|
| 1 | `RmucSubGameStatus` | `/game_status` | `game_status`, `now_ms` |
| 2 | `RmucSubRobotStatus` | `/robot_status` | `robot_status` |
| 3 | `RmucSubRFIDStatus` | `/rfid_status` | `rfid_status` |
| 4 | `RmucSubRobotPosition` | `/robot_position` | `pose_x`, `pose_y`, `pose_yaw`, `is_at_nav_goal`, `pose` |
| 5 | `SubRadarTracks` | `/radar/enemy_tracks` | `radar_tracks` |
| 6 | `RmucSubSentryDecisionStatus` **(NEW P0)** | `/sentry_decision_status` | `sentry_decision_status` |
| 7 | `RmucSubRobotBuff` **(NEW P0)** | `/robot_buff` | `robot_buff` |
| 8 | `RmucSubProjectileAllowance` **(NEW P0)** | `/projectile_allowance` | `projectile_allowance` |
| 9 | `RmucSubFieldStatus` **(NEW P0)** | `/field_status` | `field_status` |
| 10 | `RmucSubEnemyMark` **(NEW P0)** | `/enemy_mark` | `enemy_mark` |
| 11 | `RmucSubTeamPositions` **(NEW P0)** | `/team_positions` | `team_positions` |
| 12 | `RmucSubTeamHP` **(NEW P0)** | `/team_hp` | `team_hp` |

- **返回值**：正常情况下返回 `SUCCESS`
- **实现要点**：
  - **非阻塞**：每 tick 尽快返回 SUCCESS，内部缓存最新消息
  - 若消息长时间未更新，可设置降级标志供 `ParseSentryBlackboard` 使用
  - P0 新增的 7 个话题为 2026 赛季裁判系统新协议，提供增益、场地、经济、队伍等实时信息

---

### 3.2 黑板初始化与解析节点（3 个）

#### 3.2.1 InitSentryConfig（#13）

- **类型**：Action（SyncActionNode，输出配置到黑板）
- **输出端口**：
  - 各关键点坐标：`home_x/y` / `defend_anchor_x/y` / `base_buff_x/y` / `outpost_buff_x/y` / `fortress_x/y` / 高地 / 巡逻点等
  - **NEW（重命名）**：`supply_zone_x/y`（原 `supply_x/y`）
  - **NEW**：`supply_zone_x/y`（补给站导航目标点）
  - 阈值：`hp_critical` / `hp_low` / `hp_safe`、`heat_high` / `heat_critical`、`ammo_low` / `ammo_target` 等
  - **NEW**：`heal_wait_ms`（回血等待时间）、`heal_min_ratio`（最低回血比例）、`search_timeout_ms`（搜索超时时间）
- **返回值**：返回 `SUCCESS`
- **实现要点**：
  - 默认值均为 0.0 或保守阈值；**务必根据实际地图坐标系与机器人能力标定**
  - `arrive_radius` 建议与导航定位精度和模块卡死区大小匹配
  - `supply_zone_x/y` 已从 `supply_x/y` 重命名，以反映补给区增益点的实际语义

#### 3.2.2 InitCmdState（#14）

- **类型**：Action（SyncActionNode，初始化指令状态）
- **inout 端口**：`cmd_state`（内部指令状态，维护单调计数等）
- **输出端口**：`allow_ammo_target`（初始允许发弹量目标）
- **返回值**：返回 `SUCCESS`
- **实现要点**：
  - 在行为树启动时执行一次，初始化 `cmd_state` 的计数器与边沿触发状态

#### 3.2.3 ParseSentryBlackboard（#15）

- **类型**：Action（SyncActionNode，解析裁判系统与感知，派生高层语义）
- **输入端口（14 个原始消息）**：
  - 原有 7 个：`game_status` / `robot_status` / `radar_tracks` / `pose_x` / `pose_y` / `now_ms` / `rfid_status`
  - **NEW P0 — 7 个新话题输入**：`sentry_decision_status` / `robot_buff` / `projectile_allowance` / `field_status` / `enemy_mark` / `team_positions` / `team_hp`
- **输出端口（~70+ 个黑板字段）**：
  - 原有：`{game.remain_s}` / `{game.elapsed_s}`、`{hp.*}`、`{heat.cur}`、`{ammo.*}`、`{economy.*}`、`{state.*}`、`{combat.*}`、`{threat.*}` 等
  - **NEW P0 输出**：
    - 哨兵决策状态字段（来自 `sentry_decision_status`）
    - 增益数值：`buff_cool_value` / `buff_defense_pct` / `buff_vulnerability_pct`
    - 场地状态：各增益点占领/活跃状态
    - 敌方易伤：`enemy_hero_vuln` / `enemy_engi_vuln` / `enemy_infantry3_vuln` / `enemy_infantry4_vuln` / `enemy_sentry_vuln`
    - 队伍血量：各队友当前血量
  - **NEW P1 输出**：
    - `respawn_invincible`（复活无敌状态）
    - `is_power_boosted`（能量增强状态）
    - `cumulative_instant_count`（累计立即复活次数）
- **返回值**：成功解析返回 `SUCCESS`
- **实现要点**：
  - 统一做：**脱战判定**、**虚弱判定**、**基地威胁评估**、**目标筛选预处理**
  - `stage_remain_time` 直接从 `RMUCGameStatus` 读取；`elapsed` = `420 - remain`
  - P0 新增输入极大扩展了可用于决策的信息，建议在此节点完成所有增益/场地/队伍状态的预处理

---

### 3.3 决策节点（5 个）

#### 3.3.1 DecidePosture（#16）

- **类型**：Action（决定哨兵姿态）
- **输入端口**：`hp_cur` / `hp_max`、`heat_cur`、`has_target`、`base_threat`、`is_disengaged`、`stage_elapsed_time`、`now_ms`
  - **NEW 输入**：`current_posture`（当前姿态）、`buff_cool_value`（冷却增益）、`buff_defense_pct`（防御增益%）、`buff_vulnerability_pct`（易伤增益%）、`ammo_allow`（允许发弹量）
- **输出端口**：
  - `posture_out` (int)

| 值 | 姿态 |
|----|------|
| 1 | 进攻姿态 |
| 2 | 防御姿态 |
| 3 | 移动姿态 |

- **返回值**：返回 `SUCCESS`（仅输出姿态）
- **实现要点**：
  - 内置 **5 秒** 切换冷却，避免频繁抖动
  - 增益信息（冷却/防御/易伤百分比）可直接影响评分权重
  - 可在内部记录各姿态累计时间，超过 3 分钟后按规则降低该姿态收益
  - 三个 score 输出便于 Groot2 可视化调试


- **类型**：Action（Groot2 可视化专用，行为树 XML 中已注释）
- **输出端口**：`best_posture`
- **说明**：仅用于 Groot2 实时监控界面显示姿态评分；正式运行时注释掉以节省资源

#### 3.3.3 DecideEconomyCmd（#18）

- **类型**：Action（决定远程补血/补弹与允许发弹量目标）
- **输入端口**：
  - 原有：`hp_cur` / `hp_max`、`ammo_allow` / `ammo_target` / `ammo_low`、`is_disengaged`、`can_remote_heal` / `can_remote_ammo`、`team_coins`、`stage_remain_time`、`base_threat`
  - **NEW 输入**：`instant_respawn_cost`（立即复活费用）、`cumulative_instant_count`（累计立即复活次数）、`base_hp`（基地血量）、`fortress_ammo`（堡垒储备弹药）、`remote_heal_count` / `remote_ammo_count`（远程兑换次数）
- **输出端口**：`allow_ammo_target`（单调不减）、`trigger_remote_ammo` (0/1)、`trigger_remote_hp` (0/1)、`enable_big_energy` (0/1)
- **返回值**：返回 `SUCCESS`
- **实现要点**：
  - 远程补血/补弹必须满足：**脱战** + `can_remote_*` + **金币足够** + 预计 6 秒内生存概率高
  - `allow_ammo_target` 按协议要求必须 **单调递增**
  - 需考虑立即复活费用与累计次数，避免金币不足时错误触发
  - 堡垒储备弹药量可作为补弹决策的参考

#### 3.3.4 DecideRespawnCmd（#19）

- **类型**：Action（决定确认复活 / 兑换立即复活）
- **输入端口**：
  - 原有：`is_dead`、`robot_status`、`team_coins`、`stage_remain_time`、`base_hp_cur` / `base_hp_max`、`base_threat`
  - **NEW 输入**：`can_free_respawn`（是否可免费复活）、`can_instant_respawn`（是否可立即复活）、`instant_respawn_cost`（立即复活费用）
- **inout 端口**：`cumulative_instant_count` **(NEW)**（累计立即复活次数，每次立即复活 +1）
- **输出端口**：`confirm_respawn` (0/1)、`confirm_instant_respawn` (0/1)
- **返回值**：返回 `SUCCESS`
- **实现要点**：
  - 优先级：若终局且守家压力大且金币允许→倾向 **立即复活**；否则走正常读条复活并尽快触卡解除虚弱
  - `cumulative_instant_count` 作为 inout，复活节点自行递增并写回黑板
  - 注意协议处理顺序：服务器按从低位到高位依次处理 bit 指令

#### 3.3.5 SentryCmdMux（#20，0x0120）

- **类型**：Action（打包并发送"哨兵自主决策指令"到裁判系统，5Hz）
- **输入端口**：`posture`、`confirm_respawn`、`confirm_instant_respawn`、`allow_ammo_target`、`trigger_remote_ammo`、`trigger_remote_hp`、`enable_big_energy`
- **inout 端口**：`cmd_state`（内部状态，维护单调计数与边沿触发）
- **返回值**：发送成功返回 `SUCCESS`；发送失败可返回 `FAILURE`

**协议 bit 布局**：

| Bit 范围 | 功能 |
|----------|------|
| bit 0 | 确认复活 |
| bit 1 | 确认兑换立即复活 |
| bit 2–12 | 允许发弹量目标值（单调递增） |
| bit 13–16 | 远程兑换发弹量请求次数（单调递增，每次 +1） |
| bit 17–20 | 远程兑换血量请求次数（单调递增，每次 +1） |
| bit 21–22 | 姿态（1 进攻 / 2 防御 / 3 移动） |
| bit 23 | 大能量机关确认（1 确认） |

- **实现要点**：
  - `trigger_remote_ammo` / `trigger_remote_hp` 建议做 **上升沿触发**：由 0→1 时把计数 +1 并发送
  - `allow_ammo_target` 建议直接写"累计目标"，由节点内部保证单调与裁判系统一致性

---

### 3.4 导航动作节点（4 个）

#### 3.4.1 SendGoal（#21）

- **类型**：Action（发送导航目标点）
- **输入端口**：`goal_x` / `goal_y`、`frame_id`（坐标系名称）、`action_name`（Nav2 Action 名称）
- **返回值**：发送成功返回 `SUCCESS`；Action Server 不可用返回 `FAILURE`
- **实现要点**：调用 Nav2 的 `NavigateToPose` Action，非阻塞发送

#### 3.4.2 CancelNavGoal（#22）

- **类型**：Action（取消当前导航目标）
- **输入端口**：`action_name`（需取消的 Nav2 Action 名称）
- **返回值**：取消成功返回 `SUCCESS`
- **实现要点**：用于战术打断——收到更高优先级任务时取消导航

#### 3.4.3 MoveAround（#23）

- **类型**：Action（在当前位置附近随机移动）
- **输入端口**：`expected_nearby_goal_count`（附近目标数量）、`expected_dis`（期望移动距离）、`message`（`pose` 消息）
- **返回值**：返回 `SUCCESS` / `RUNNING`
- **实现要点**：用于巡逻/探索，在当前位置一定范围内随机生成导航点

#### 3.4.4 KeepRunning（#24）

- **类型**：Action（始终返回 RUNNING）
- **端口**：无输入输出端口
- **返回值**：始终返回 `RUNNING`
- **实现要点**：用作 WhileDoElse / ReactiveFallback 的占位子节点，保持父节点持续 tick

---

### 3.5 交战节点（3 个）

#### 3.5.1 SelectBestTarget（#25）

- **类型**：Action（从雷达跟踪列表选择最优目标）
- **输入端口**：
  - `radar_tracks`（雷达跟踪数据）、`pose`（自身位姿）、`base_x` / `base_y`（基地坐标）
  - **NEW 输入**：`enemy_hero_vuln` / `enemy_engi_vuln` / `enemy_infantry3_vuln` / `enemy_infantry4_vuln` / `enemy_sentry_vuln`（敌方各兵种易伤状态）
- **输出端口**：`out_target`（选中的目标字符串）
- **返回值**：找到目标返回 `SUCCESS`；无有效目标返回 `FAILURE`
- **实现要点**：
  - 综合权重：距离、对基地威胁、血量、**易伤状态（NEW）**
  - 易伤目标应获得更高优先级（伤害倍率更高，击杀效率更好）

#### 3.5.2 AimAtTarget（#26）

- **类型**：Action（云台对准目标）
- **输入端口**：`target`（目标字符串，由 SelectBestTarget 输出）
- **返回值**：对准过程中返回 `RUNNING`；完成返回 `SUCCESS`

#### 3.5.3 FireBurst（#27）

- **类型**：Action（按节奏开火以控热）
- **输入端口**：`burst_ms`（单次连射持续毫秒）、`pause_ms`（连射间暂停毫秒）
- **返回值**：开火过程中返回 `RUNNING`；完成返回 `SUCCESS`
- **实现要点**：结合热量模型与上限，避免超限惩罚

---

### 3.6 机器人控制节点（1 个）

#### 3.6.1 RmucRobotControl（#28）

- **类型**：Action（发布底层控制指令）
- **输入端口**：`stop_gimbal_scan` (bool)、`chassis_spin` (bool)、`fire_enable` (bool)
- **返回值**：返回 `SUCCESS`
- **实现要点**：
  - `stop_gimbal_scan=true` → 云台停止扫描（占点/回血时使用）
  - `chassis_spin=true` → 底盘陀螺/旋转以增加生存能力
  - `fire_enable=false` → 禁止开火（非比赛阶段/虚弱状态）

---

### 3.7 导航选择节点（8 个）

#### 3.7.1 SelectObjective（#29）

- **类型**：Action（综合战局选择导航目标）
- **输入端口**：
  - `pose_x` / `pose_y`、`stage_elapsed_time`、`hp_cur` / `hp_max`、`ammo_allow`、`base_hp_cur` / `base_hp_max`、`base_deficit`、`outpost_alive`、`base_threat`
  - **NEW 输入**：`field_supply_status` / `field_base_buff_status` / `field_outpost_buff_status` / `field_fortress_status` / `field_enemy_fortress_status`（5 个场地状态）
  - **NEW 输入**：`fortress_ammo`（堡垒储备弹药量）
  - **NEW 输入**：所有 8 组关键点坐标（home/supply/base_buff/outpost_buff/fortress/highland/trapezoid/patrol 的 x/y）
- **输出端口**：`goal_x` / `goal_y`、`objective_name`（目标名称字符串）
- **返回值**：找到目标返回 `SUCCESS`；找不到返回 `FAILURE`
- **实现要点**：
  - 场地状态可直接影响目标价值评估（已被占领/失效的增益点价值降低）
  - 建议固定优先级：**守家/堡垒 > 中央高地 > 梯形高地 > 其它地形/巡逻**

#### 3.7.2 HoldObjective（#30）

- **类型**：Action（占点保持一段时间）
- **输入端口**：`hold_ms`（保持毫秒数）、`base_threat` (bool)、`has_target` (bool)
- **返回值**：等待中返回 `RUNNING`；计时结束或 `base_threat` / `has_target` 触发提前结束返回 `SUCCESS`

#### 3.7.3 WaypointPatrol（#31）

- **类型**：Action（巡逻点循环巡逻）
- **输入端口**：`pose`（当前位姿）+ 3 个巡逻点（`wp1_x/y`、`wp2_x/y`、`wp3_x/y`）
- **输出端口**：`goal_x` / `goal_y`（下一个巡逻目标）
- **返回值**：返回 `SUCCESS`
- **实现要点**：按顺序循环遍历巡逻点；到达当前点后自动切换下一个

#### 3.7.4 SelectNearestDispelCard（#32）

- **类型**：Action（选择最近的解除虚弱模块卡位置）
- **输入端口**：`pose_x` / `pose_y`、`supply_zone_x/y`、`base_buff_x/y`、`outpost_buff_x/y`
- **输出端口**：`goal_x` / `goal_y`
- **返回值**：找到目标返回 `SUCCESS`；找不到返回 `FAILURE`
- **实现要点**：选择距离最近的可用模块卡位置，用于虚弱状态解除

#### 3.7.5 SelectNearestResupplyStation（#33）

- **类型**：Action（选择最近的补给站位置）
- **输入端口**：`pose_x` / `pose_y`、`supply_zone_x/y`、`base_buff_x/y`、`outpost_buff_x/y`
- **输出端口**：`goal_x` / `goal_y`
- **返回值**：找到目标返回 `SUCCESS`；找不到返回 `FAILURE`

#### 3.7.6 SelectSafeRetreatGoal（#34）

- **类型**：Action（选择安全撤退目标点）
- **输入端口**：`pose_x` / `pose_y`、`supply_zone_x/y`、`defend_anchor_x/y`
- **输出端口**：`goal_x` / `goal_y`
- **返回值**：返回 `SUCCESS`
- **实现要点**：综合当前位置和补给区/防守锚点，选择最安全的撤退方向

#### 3.7.7 HoldAndHeal（#35）

- **类型**：Action（在补给区等待回血）
- **输入端口**：`hp_cur` / `hp_max` / `hp_safe`、`stage_elapsed_time`、`is_disengaged`
- **返回值**：等待中返回 `RUNNING`；血量恢复至 `hp_safe` 返回 `SUCCESS`
- **实现要点**：
  - 4 分钟后脱战可获 25%/s 高速回血
  - 等待过程中可允许云台扫描，但应避免误触发射导致脱战失效

#### 3.7.8 HoldForSupplyAmmoTick（#36）

- **类型**：Action（在补给区等待弹药补给刷新）
- **输入端口**：`stage_elapsed_time`、`ammo_allow`、`ammo_target`
- **返回值**：等待中返回 `RUNNING`；达到 `ammo_target` 或补给刷新完成返回 `SUCCESS`
- **实现要点**：补给区每分钟占领一次累计 +100 允许发弹量

---

### 3.8 复活恢复节点（4 个，NEW）

#### 3.8.1 RmucNavControlCmd（#37）

- **类型**：Action（发布导航控制指令）
- **输入端口**：`cmd_type` (int: 1=导航, 3=停止)、`emergency_stop` (bool)
- **返回值**：返回 `SUCCESS`
- **实现要点**：
  - `cmd_type=1`：启动导航模式
  - `cmd_type=3`：停止所有导航运动
  - `emergency_stop=true`：紧急停车
  - 通常在复活后使用：先停止（确保安全），再启动导航前往触卡解除虚弱

#### 3.8.2 RmucWaitAndHeal（#38）

- **类型**：Action（等待并监控回血进度）
- **输入端口**：`topic_name`（话题名）、`now_ms`（当前时间）、`hp_cur` / `hp_max`（当前/上限血量）、`heal_wait_ms`（等待时间阈值）、`heal_min_ratio`（最低回血比例，低于此值判定回血失败）
- **inout 端口**：`heal_start_ms`（回血起始时间戳，首次写入后续读取）
- **返回值**：回血中返回 `RUNNING`；回血完成返回 `SUCCESS`；超时或回血不足返回 `FAILURE`
- **实现要点**：
  - 首次 tick 记录 `heal_start_ms`，后续 tick 检查回血进度
  - 若超过 `heal_wait_ms` 但血量恢复比例未达 `heal_min_ratio`，判定回血失败并返回 FAILURE

#### 3.8.3 InitSearchTimerIfNeeded（#39）

- **类型**：Action（初始化搜索计时器）
- **inout 端口**：`search_start_ms`（搜索开始时间戳）
- **返回值**：返回 `SUCCESS`
- **实现要点**：若 `search_start_ms` 未初始化（或为 0），则设置为当前时间；否则保持不变

#### 3.8.4 RmucMicroSearchSupplyCard（#40）

- **类型**：Action（微动搜索补给区模块卡）
- **inout 端口**：`search_start_ms`（搜索开始时间戳）
- **输入端口**：`timeout_ms`（搜索超时毫秒）、`rfid_status`（RFID 状态）
- **返回值**：搜索中返回 `RUNNING`；找到卡返回 `SUCCESS`；超时返回 `FAILURE`
- **实现要点**：
  - 复活后虚弱状态下，在补给区附近微小幅度移动以寻找模块卡
  - 超过 `timeout_ms` 未找到卡则放弃，由上层切换到其他解除虚弱策略

---

### 3.9 条件节点（16 个）

> 所有条件节点：满足条件返回 `SUCCESS`，不满足返回 `FAILURE`。

| # | 节点 | 输入端口 | 判定逻辑 |
|:-:|:---|:---|:---|
| 41 | `RmucIsGameTime` | `message`(game_status), `game_progress`, `lower_remain_time`, `higher_remain_time` | 比赛阶段为指定 `game_progress` 且剩余时间在 `[lower, higher]` 范围内 |
| 42 | `RmucIsDead` | `message`(robot_status) | 机器人处于死亡状态 |
| 43 | `IsWeakness` | `robot_status` | 检查发射机构断电(`shooter_power_output`) + 存活 + 非热量超限 + 非零弹丸；内置 20 帧去抖 |
| 44 | `RmucIsHPBelow` | `message`(robot_status), `hp_threshold` | 当前血量低于阈值 |
| 45 | `IsAmmoBelow` | `ammo_allow`, `ammo_low` | 允许发弹量低于阈值 |
| 46 | `IsCriticalState` | `hp_cur`, `hp_critical`, `heat_cur`, `heat_critical` | 血量或热量达到危急阈值 |
| 47 | `IsBaseThreatened` | `base_threat`, `base_hp_cur`, `base_hp_max`, `enemy_near_base_radius` + **NEW**: `outpost_alive` | 基地受到威胁（考虑前哨站存活状态） |
| 48 | `HasValidTarget` | `has_target`, `best_target` | 存在有效交战目标 |
| 49 | `IsCombatAllowed` | `robot_status`(NOT is_weak), `ammo_allow`, `heat_cur`, `heat_high`, `hp_cur`, `hp_low` | 综合判定是否允许进入交战（非虚弱+弹药+热量+血量） |
| 50 | `IsFireWindowOk` | `heat_cur`, `heat_high`, `ammo_allow`, `robot_status` + **NEW**: `current_posture`, `buff_cool_value`, `buff_vulnerability_pct`, `ammo_conserve`(默认30) | 开火窗口门控：热量安全+弹药充足+非虚弱；NEW: 根据姿态和增益动态调整阈值 |
| 51 | `IsZoneCardDetected` | `zone`(字符串参数), `rfid_status`, `robot_status` | 检测指定区域的模块卡（SUPPLY/BASE_BUFF/OUTPOST_BUFF/FORTRESS 等） |
| 52 | `ShouldChassisSpin` **(NEW P4)** | `pose_x`, `pose_y`, `goal_x`, `goal_y`, `arrive_radius`, `current_posture`, `is_power_boosted`, `force_spin` | 判定是否应启用底盘旋转：考虑距目标距离、当前姿态、能量增强状态等 |
| 53 | `IsAnyDispelCardDetected` | `rfid_status`, `robot_status` | 检测到任意可占领的补给区/基地/前哨站卡（用于虚弱解除） |
| 54 | `IsAtGoal` | `pose_x`, `pose_y`, `goal_x`, `goal_y`, `arrive_radius` | 当前位置在目标点 `arrive_radius` 范围内 |
| 55 | `RmucIsAtNavGoal` | `is_at_nav_goal` | 导航系统报告已到达目标（直接读黑板布尔值） |
| 56 | `RmucIsSupplyCardDetected` | `rfid_status` | 检测到补给区模块卡（专用于补给区交互） |

---

### 3.10 装饰器节点（1 个）

#### 3.10.1 RateController（#57）

- **类型**：Decorator（频率控制装饰器）
- **输入端口**：`hz` (double，执行频率)
- **功能**：限制子节点的 tick 频率，每 `1/hz` 秒才允许子节点执行一次
- **返回值**：当子节点被 tick 时，透传子节点的返回值；未到执行时间时返回上一次的缓存结果
- **实现要点**：
  - 用于 `SentryCmdMux`（5Hz）等需要固定频率发送的节点
  - 避免高频循环导致不必要的通信负载

## 附录

### A. 行为树集成建议

1. **CommandHub 子树** 建议始终运行，保证姿态与远程兑换请求在任何战术分支中都能持续更新
2. 若工程已有"导航到点 + 到达判定 + 微动触卡"封装，可直接替换 `HealPlan` / `AmmoPlan` 的相关节点
3. **map 坐标与关键点** 必须在上场前标定；建议把 `InitSentryConfig` 的默认值替换为 YAML / 参数服务器加载
4. 模块卡可能存在延迟与死区，占点相关 Condition 节点 **务必做滤波去抖**
5. **复活恢复流程 (NEW)**：死亡 → DecideRespawnCmd → 复活后 → RmucNavControlCmd(停止) → 导航到最近触卡点 → 微动搜索 → 解除虚弱
6. **P0 新增话题** 提供了大量决策信息（增益/场地/队伍/经济），建议在 `ParseSentryBlackboard` 中统一预处理

### B. 消息类型与话题对照

| 消息类型 | 话题 | 方向 | 备注 |
|:---|:---|:---:|:---|
| `RMUCGameStatus` | `/game_status` | 📥 | 比赛阶段/剩余时间 |
| `RMUCRobotStatus` | `/robot_status` | 📥 | 机器人状态/血量/热量 |
| `RMUCRFIDStatus` | `/rfid_status` | 📥 | RFID 模块卡检测 |
| `RMUCRobotPosition` | `/robot_position` | 📥 | 定位/导航到达 |
| `RMUCEnemyTracks` | `/radar/enemy_tracks` | 📥 | 雷达目标跟踪 |
| `RMUCSentryDecisionStatus` | `/sentry_decision_status` | 📥 | **(NEW P0)** 哨兵决策状态反馈 |
| `RMUCRobotBuff` | `/robot_buff` | 📥 | **(NEW P0)** 增益数值 |
| `RMUCProjectileAllowance` | `/projectile_allowance` | 📥 | **(NEW P0)** 弹丸配额 |
| `RMUCFieldStatus` | `/field_status` | 📥 | **(NEW P0)** 场地增益点状态 |
| `RMUCEnemyMark` | `/enemy_mark` | 📥 | **(NEW P0)** 敌方标记/易伤 |
| `RMUCTeamPositions` | `/team_positions` | 📥 | **(NEW P0)** 队伍位置 |
| `RMUCTeamHP` | `/team_hp` | 📥 | **(NEW P0)** 队伍血量 |
| `RMUCSentryCmd` | `/sentry_cmd` | 📤 | 哨兵自主决策指令 (0x0120) |
| `RMUCRobotControl` | `/robot_control` | 📤 | 底盘/云台/发射控制 |
| `RMUCNavControlCmd` | `/nav_control_cmd` | 📤 | **(NEW)** 导航控制指令 |

> 详细的拆分说明、踩坑记录见 `RMUC_Msg_Split_Doc.ipynb`。

### C. 自定义节点完整索引

| # | 节点名 | 类型 | 章节 |
|:-:|:---|:---|:---|
| 1 | RmucSubGameStatus | Sub Action | 3.1 |
| 2 | RmucSubRobotStatus | Sub Action | 3.1 |
| 3 | RmucSubRFIDStatus | Sub Action | 3.1 |
| 4 | RmucSubRobotPosition | Sub Action | 3.1 |
| 5 | SubRadarTracks | Sub Action | 3.1 |
| 6 | RmucSubSentryDecisionStatus | Sub Action (NEW) | 3.1 |
| 7 | RmucSubRobotBuff | Sub Action (NEW) | 3.1 |
| 8 | RmucSubProjectileAllowance | Sub Action (NEW) | 3.1 |
| 9 | RmucSubFieldStatus | Sub Action (NEW) | 3.1 |
| 10 | RmucSubEnemyMark | Sub Action (NEW) | 3.1 |
| 11 | RmucSubTeamPositions | Sub Action (NEW) | 3.1 |
| 12 | RmucSubTeamHP | Sub Action (NEW) | 3.1 |
| 13 | InitSentryConfig | Action | 3.2.1 |
| 14 | InitCmdState | Action | 3.2.2 |
| 15 | ParseSentryBlackboard | Action | 3.2.3 |
| 16 | DecidePosture | Action | 3.3.1 |
| 18 | DecideEconomyCmd | Action | 3.3.3 |
| 19 | DecideRespawnCmd | Action | 3.3.4 |
| 20 | SentryCmdMux | Action | 3.3.5 |
| 21 | SendGoal | Action | 3.4.1 |
| 22 | CancelNavGoal | Action | 3.4.2 |
| 23 | MoveAround | Action | 3.4.3 |
| 24 | KeepRunning | Action | 3.4.4 |
| 25 | SelectBestTarget | Action | 3.5.1 |
| 26 | AimAtTarget | Action | 3.5.2 |
| 27 | FireBurst | Action | 3.5.3 |
| 28 | RmucRobotControl | Action | 3.6.1 |
| 29 | SelectObjective | Action | 3.7.1 |
| 30 | HoldObjective | Action | 3.7.2 |
| 31 | WaypointPatrol | Action | 3.7.3 |
| 32 | SelectNearestDispelCard | Action | 3.7.4 |
| 33 | SelectNearestResupplyStation | Action | 3.7.5 |
| 34 | SelectSafeRetreatGoal | Action | 3.7.6 |
| 35 | HoldAndHeal | Action | 3.7.7 |
| 36 | HoldForSupplyAmmoTick | Action | 3.7.8 |
| 37 | RmucNavControlCmd | Action (NEW) | 3.8.1 |
| 38 | RmucWaitAndHeal | Action (NEW) | 3.8.2 |
| 39 | InitSearchTimerIfNeeded | Action (NEW) | 3.8.3 |
| 40 | RmucMicroSearchSupplyCard | Action (NEW) | 3.8.4 |
| 41 | RmucIsGameTime | Condition | 3.9 |
| 42 | RmucIsDead | Condition | 3.9 |
| 43 | IsWeakness | Condition | 3.9 |
| 44 | RmucIsHPBelow | Condition | 3.9 |
| 45 | IsAmmoBelow | Condition | 3.9 |
| 46 | IsCriticalState | Condition | 3.9 |
| 47 | IsBaseThreatened | Condition | 3.9 |
| 48 | HasValidTarget | Condition | 3.9 |
| 49 | IsCombatAllowed | Condition | 3.9 |
| 50 | IsFireWindowOk | Condition | 3.9 |
| 51 | IsZoneCardDetected | Condition | 3.9 |
| 52 | ShouldChassisSpin | Condition (NEW) | 3.9 |
| 53 | IsAnyDispelCardDetected | Condition | 3.9 |
| 54 | IsAtGoal | Condition | 3.9 |
| 55 | RmucIsAtNavGoal | Condition | 3.9 |
| 56 | RmucIsSupplyCardDetected | Condition | 3.9 |
| 57 | RateController | Decorator | 3.10 |